In [38]:
import torch

# probs: softmax 결과 (shape: [1,7])
def depression_score(probs):
    happiness, surprise, neutral, fear, disgust, anger, sadness = probs.tolist()[0]

    POS = happiness + surprise
    NEG_w = (0.4 * sadness) + (0.3 * neutral) + (0.1 * fear) + (0.1 * disgust) + (0.1 * anger)

    score = NEG_w / (NEG_w + POS + 1e-8)
    return score


In [ ]:
from sklearn.metrics import classification_report, f1_score
from datetime import datetime
import os, torch
from data_model.BuildModel import BuildModel
from data_model.ModelType import ModelType
import torchvision.models as M
import torch
from util_tool import resolve_path
from torchvision import transforms as T
from data_model.GlobalVariable import ImageVariable
from pre_process.img_preprocess import TrimBorder, compute_mean_std, TimeMask, FreqMask
from pathlib import Path
import numpy as np
np.set_printoptions(precision=4, suppress=True)

device = "cuda:0" if torch.cuda.is_available() else "cpu"
if os.name == "posix":  # Linux, macOS
    path = "/home/wanted-1/PotenupWorkspace/aug-project5/jin_sup/model/model_ConvNeXt_Small_Weights.IMAGENET1K_V1___08-25_20-53-16.pth"
elif os.name == "nt":  # Windows
    path = "C:\\PythonProject\\aug-08month_project5\\jin_sup\\model\\model_ConvNeXt_Small_Weights.IMAGENET1K_V1___08-25_20-53-16.pth"


checkpoint = torch.load(path, map_location="cpu")

y_true, y_pred = [], []
model = BuildModel.get_model(ModelType.CONVNEXT_SMALL,
                             M.ConvNeXt_Small_Weights.IMAGENET1K_V1,
                               7 , checkpoint, device)

mean = ImageVariable.MEAN
std = ImageVariable.STD
norm = T.Normalize(mean=[mean, mean, mean], std=[std, std, std])

from PIL import Image
tf = T.Compose([
    TrimBorder(0),
    T.Grayscale(3),
    T.Resize(ImageVariable.IMAGE_SIZES, antialias=True),
    T.CenterCrop(ImageVariable.IMAGE_SIZES),
    T.ToTensor(),
    norm,
])

import glob
root_folder = r"C:\\PythonProject\\aug-08month_project5\\jin_sup\\my_test"
png_files = glob.glob(root_folder + "/**/*.png", recursive=True)

preds = []
confs = [] 

for png_path in png_files:
    img = Image.open(png_path).convert("RGB")
    x = tf(img).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1)

        print(depression_score(probs))
        arr = probs.cpu().numpy()

        print(type(arr))
        print(arr)
        
        conf, pred = torch.max(probs, dim=1)
        preds.append(pred.item())
        confs.append(conf.item())

        print(conf)
        print(pred)

0.005191483696632484
tensor([0.9701])
tensor([0])
0.9932293250274778
tensor([0.6493])
tensor([6])
0.9991215844711909
tensor([0.9992])
tensor([6])
